## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("BookPreviousApp") \
    .master("spark://spark-master:7077") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/17 16:36:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
credit = spark.read \
    .parquet("/data/processed/credit_card")

credit.createOrReplaceTempView("credit")

# Contagem de linhas e colunas
num_rows = credit.count()
num_columns = len(credit.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

credit.show(5, truncate=False)

Quantidade de linhas: 3840312
Quantidade de variaveis (colunas): 24
+-------------+----------+----------------------+----------------------------------+-----------------------------------+-------------------------------+-------------------------------------+-----------------------------------+----------------------------------+------------------------------+------------------------------------+-----------------------------------+------------------------+-------------------------------+-----------------------------------+-------------------------------+-------------------------------------+-----------------------------------+------------------------------------+------------------------------+-----------------+---------------------+-------------------+-----------------+
|PK_AGG_CREDIT|SK_ID_CURR|FVL_AMT_BALANCE_CREDIT|FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT|FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT|FVL_AMT_DRAWINGS_CURRENT_CREDIT|FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT|FVL_AMT_DRAWINGS_POS_CURRENT_CR

In [4]:
pos_cash = spark.read \
    .parquet("/data/processed/pos_cash")

pos_cash.createOrReplaceTempView("pos_cash")

# Contagem de linhas e colunas
num_rows = pos_cash.count()
num_columns = len(pos_cash.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

pos_cash.show(5, truncate=False)

[Stage 14:===================================>                    (12 + 6) / 19]

Quantidade de linhas: 10001358
Quantidade de variaveis (colunas): 9
+----------+----------+----------------------+-----------------------------+---------------------------+--------------+------------------+-------------------+--------------+
|PK_AGG_POS|SK_ID_CURR|FVL_CNT_INSTALMENT_POS|FVL_CNT_INSTALMENT_FUTURE_POS|FC_NAME_CONTRACT_STATUS_POS|FVL_SK_DPD_POS|FVL_SK_DPD_DEF_POS|PK_DATREF_POS      |PK_DATPART_POS|
+----------+----------+----------------------+-----------------------------+---------------------------+--------------+------------------+-------------------+--------------+
|2137736   |355067    |6.0                   |0.0                          |Completed                  |0             |0                 |2022-12-01 00:00:00|202212        |
|2094457   |411079    |12.0                  |9.0                          |Active                     |0             |0                 |2022-12-01 00:00:00|202212        |
|1556649   |288150    |6.0                   |2.0             

In [9]:
installments = spark.read \
    .parquet("/data/processed/installments")

installments.createOrReplaceTempView("installments")

# Contagem de linhas e colunas
num_rows = installments.count()
num_columns = len(installments.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

installments.show(5, truncate=False)

Quantidade de linhas: 13605401
Quantidade de variaveis (colunas): 7
+-------------------+----------+------------------------------+--------------------------+-----------------------+-------------------+------------------+
|PK_AGG_INSTALLMENTS|SK_ID_CURR|FVL_DAYS_ENTRY_PAYMENT_INSTALM|FVL_AMT_INSTALMENT_INSTALM|FVL_AMT_PAYMENT_INSTALM|PK_DATREF_INSTALM  |PK_DATPART_INSTALM|
+-------------------+----------+------------------------------+--------------------------+-----------------------+-------------------+------------------+
|1135379            |366142    |-209.0                        |35758.485                 |35758.485              |2023-05-10 00:00:00|202305            |
|1849293            |308244    |-210.0                        |2021.445                  |2021.445               |2023-05-05 00:00:00|202305            |
|2262465            |388494    |-214.0                        |14052.375                 |14052.375              |2023-05-22 00:00:00|202305            |
|2482592

In [5]:
previous_app = spark.read \
    .parquet("/data/processed/previous_app")

previous_app.createOrReplaceTempView("previous_app")

# Contagem de linhas e colunas
num_rows = previous_app.count()
num_columns = len(previous_app.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

previous_app.show(5, truncate=False)

25/07/17 16:42:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Quantidade de linhas: 1670214
Quantidade de variaveis (colunas): 36
+---------------+----------+------------------------------+------------------------+----------------------------+-----------------------+-----------------------------+----------------------------+--------------------------------------+------------------------------------+-----------------------------------+------------------------------+----------------------------------+-------------------------------------+----------------------------------+--------------------------------+-----------------------------+------------------------------+---------------------------+----------------------------+-------------------------------+--------------------------+------------------------+-----------------------------+------------------------+--------------------------------+----------------------------+-------------------------------+-------------------------------+---------------------------+--------------------------------------+--

In [11]:
previous_01 = spark.sql("""
                            SELECT 
                                a.*,
                                b.*,
                                c.*
                            FROM
                                previous_app as a
                           LEFT JOIN
                                pos_cash as b ON a.PK_AGG_PREVIOUS = b.PK_AGG_POS
                           LEFT JOIN
                                installments as c ON a.PK_AGG_PREVIOUS = c.PK_AGG_INSTALLMENTS
                           LEFT JOIN
                                credit as d ON a.PK_AGG_PREVIOUS = d.PK_AGG_CREDIT 
""")

previous_01.registerTempTable("previous_01")
previous_01.count()

301991340

In [12]:
previous_01.show(5, truncate=False)

+---------------+----------+------------------------------+------------------------+----------------------------+-----------------------+-----------------------------+----------------------------+--------------------------------------+------------------------------------+-----------------------------------+------------------------------+----------------------------------+-------------------------------------+----------------------------------+--------------------------------+-----------------------------+------------------------------+---------------------------+----------------------------+-------------------------------+--------------------------+------------------------+-----------------------------+------------------------+--------------------------------+----------------------------+-------------------------------+-------------------------------+---------------------------+--------------------------------------+--------------------------+-----------------------------+-------------

In [13]:
spark.stop()